# Full Ablation for the Safeguarded LLM-in-the-Loop ETL Paper

Runs the 2×2 ablation (**circular / cross-only / rules-only / hybrid**) **plus the Reflexion
baseline** on UCI Adult and KramaBench water-quality data, then emits the pooled results
table as paste-ready LaTeX.

**Before running:**
1. GPU runtime (Kaggle: T4/P100; Colab: T4 or better).
2. A Hugging Face token with access granted to `meta-llama/Llama-3.1-8B-Instruct` and
   `mistralai/Mistral-7B-Instruct-v0.2`.
   - Kaggle: add it as a Secret named `HF_TOKEN`.
   - Colab: add it via the key icon (Secrets) as `HF_TOKEN`.

Rough budget: models load once per mode; with 5 seeds expect a few hours on a T4.
Start with `SEEDS = "1,2,3"` if constrained — 3 seeds already beats the single-seed status quo.

In [ ]:
SEEDS = "1,2,3,4,5"   # increase to "1,...,10" if GPU budget allows
REPO_ZIP_URL = "https://github.com/dhiaamejdi/LLM-in-the-Loop-Feedback-Driven-ETL/archive/refs/heads/main.zip"


In [ ]:
# Get the HF token from Kaggle or Colab secrets, or the environment
import os
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Set an HF_TOKEN secret (see instructions above)."
os.environ["HF_TOKEN"] = hf_token
print("Token loaded (not printed).")

In [ ]:
%pip -q install "transformers>=4.43" "accelerate>=0.30" "huggingface_hub>=0.23" pandas numpy
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Fetch the code (or upload the repo as a Kaggle dataset and adjust the path)
import urllib.request, zipfile, os, glob, shutil
if not os.path.exists("safeguarded_LLM_in_the_loop.py"):
    urllib.request.urlretrieve(REPO_ZIP_URL, "repo.zip")
    with zipfile.ZipFile("repo.zip") as z: z.extractall(".")
    root = glob.glob("LLM-in-the-Loop-*")[0]
    for f in ["safeguarded_LLM_in_the_loop.py", "aggregate_results.py"]:
        shutil.copy(os.path.join(root, f), ".")
    shutil.copytree(os.path.join(root, "datasets"), "datasets", dirs_exist_ok=True)
print(os.listdir("."))

In [ ]:
# === UCI Adult: 2x2 + Reflexion ===
import subprocess, sys, os
os.makedirs("runs", exist_ok=True)
for mode in ["rules-only", "circular", "reflexion", "cross-only", "hybrid"]:
    print(f"\n########## UCI / {mode} ##########")
    subprocess.run([sys.executable, "safeguarded_LLM_in_the_loop.py",
                    "--mode", mode, "--seeds", SEEDS, "--outfile-prefix", "runs/uci"], check=True)

In [ ]:
# === KramaBench water-quality: same matrix ===
import subprocess, sys
for mode in ["rules-only", "circular", "reflexion", "cross-only", "hybrid"]:
    print(f"\n########## KramaBench / {mode} ##########")
    subprocess.run([sys.executable, "safeguarded_LLM_in_the_loop.py",
                    "--mode", mode, "--seeds", SEEDS,
                    "--data-csv", "datasets/water-body-testing-2019.csv",
                    "--rules-json", "datasets/kramabench_water_rules.json",
                    "--sample-size", "1000", "--heldout-size", "100",
                    "--outfile-prefix", "runs/kb"], check=True)

In [ ]:
# === Aggregate: pooled stats + paste-ready LaTeX table ===
import subprocess, sys
subprocess.run([sys.executable, "aggregate_results.py", "runs"], check=True)
print(open("runs/ablation_table.tex").read())

## What to do with the output
1. Paste `runs/ablation_table.tex` into **Section 7.3** of the paper, replacing the
   "we defer the numbers" sentence with the table and 2–3 sentences of interpretation.
2. The key comparisons for the paper's argument:
   - **circular vs. cross-only** → the pure cross-architecture effect (H2).
   - **circular vs. reflexion** → whether self-critique alone recovers blind spots
     (H1 predicts it largely does not — this answers Reviewer 1VsF's agentic-baseline point).
   - **rules-only vs. hybrid** → what the LLM adds beyond deterministic checks.
3. Zip the `runs/` folder and add it to the repo so every number is verifiable.